#　訓練済み分類器でtestデータセットを分離


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score
import pandas as pd
from PIL import Image
import numpy as np
import os
from pathlib import Path
import random

# =========================
# 設定
# =========================
DATA_DIR = Path("/mnt/data1/Public/MedImages/DermMel")
TRAIN_IMG_DIR = DATA_DIR / "train_sep"
VAL_IMG_DIR = DATA_DIR / "valid"
TEST_IMG_DIR = DATA_DIR / "test"

# DermMel はフォルダ名がクラス名になっている構造を想定
# train_sep/
# ├── Melanoma/
# └── NotMelanoma/
# valid/
# ├── Melanoma/
# └── NotMelanoma/
# test/（ラベルなし想定）

# =========================
# データ変換（Data Augmentation）
# =========================
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# =========================
# Dataset定義
# =========================
class ImageFolderWithLabel(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        self.class_to_idx = {"NotMelanoma": 0, "Melanoma": 1}

        for cls in self.class_to_idx.keys():
            class_dir = Path(root_dir) / cls
            files = list(class_dir.glob("*.*"))
            for f in files:
                self.samples.append((f, self.class_to_idx[cls]))

        random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


# =========================
# Dataset & DataLoader
# =========================
train_dataset = ImageFolderWithLabel(TRAIN_IMG_DIR, train_transform)
val_dataset = ImageFolderWithLabel(VAL_IMG_DIR, val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

# =========================
# モデル定義
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 1)  # 2クラス→1出力 (sigmoid)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-5)

# =========================
# EarlyStopping（モデル保存機能付き）
# =========================
class EarlyStopping:
    def __init__(self, patience=5, verbose=False, delta=0, save_path="best_model.pth"):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_acc_max = 0
        self.delta = delta
        self.best_model = None
        self.save_path = save_path  # 追加

    def __call__(self, val_acc, model):
        score = val_acc
        if self.best_score is None:
            self.best_score = score
            self.best_model = model.state_dict()
            torch.save(self.best_model, self.save_path)
            if self.verbose:
                print(f"✅ Best model saved (acc: {val_acc:.4f})")
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model = model.state_dict()
            torch.save(self.best_model, self.save_path)
            if self.verbose:
                print(f"✅ Best model updated (acc: {val_acc:.4f})")
            self.counter = 0



In [2]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# モデル定義（学習時と同じ）
model = models.resnet50(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 1)  # 1ユニット出力
model = model.to(device)

# 重みロード
ckpt_path = "/mnt/data1/gotou/kaggle/path/DermMel_best_resnet50.pth"
state_dict = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print(f"Loaded model from {ckpt_path}")


/home/gotou/miniconda3/envs/env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/gotou/miniconda3/envs/env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Loaded model from /mnt/data1/gotou/kaggle/path/DermMel_best_resnet50.pth


In [3]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# =========================
# テストデータ用 Dataset
# =========================
class TestImageFolder(Dataset):
    def __init__(self, root_dir, transform=None, labeled=False):
        self.transform = transform
        self.labeled = labeled
        self.samples = []

        if labeled:
            class_to_idx = {"NotMelanoma": 0, "Melanoma": 1}
            for cls, label in class_to_idx.items():
                files = list((Path(root_dir) / cls).glob("*.*"))
                for f in files:
                    self.samples.append((f, label))
        else:
            files = list(Path(root_dir).glob("*.*"))
            self.samples = [(f, -1) for f in files]  # ラベルなし

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label, path.name


# =========================
# テストデータロード
# =========================
# labeled=True ならフォルダ構造にクラス分けあり
test_dataset = TestImageFolder(TEST_IMG_DIR, transform=val_transform, labeled=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# =========================
# 評価
# =========================
model.eval()
preds, labels, filenames = [], [], []

with torch.no_grad():
    for imgs, lbls, names in tqdm(test_loader):
        imgs = imgs.to(device)
        outputs = model(imgs).squeeze()
        probs = torch.sigmoid(outputs)
        pred = (probs > 0.5).long().cpu().numpy()

        preds.extend(pred)
        labels.extend(lbls.numpy())
        filenames.extend(names)

# =========================
# 結果出力
# =========================
if labels[0] != -1:  # ラベル付きの場合
    acc = accuracy_score(labels, preds)
    print(f"\n✅ Test Accuracy: {acc*100:.2f}%")
    print("\nConfusion Matrix:\n", confusion_matrix(labels, preds))
    print("\nClassification Report:\n", classification_report(labels, preds, target_names=["NotMelanoma", "Melanoma"]))
else:
    # ラベルなし → CSV 出力
    df = pd.DataFrame({
        "filename": filenames,
        "pred_prob": [float(p) for p in probs.cpu().numpy()],
        "pred_class": [int(c) for c in preds],
    })
    df.to_csv("DermMel_test_predictions.csv", index=False)
    print("✅ test予測結果を DermMel_test_predictions.csv に保存しました")


100%|██████████| 112/112 [00:27<00:00,  4.12it/s]


✅ Test Accuracy: 96.15%

Confusion Matrix:
 [[1746   34]
 [ 103 1678]]

Classification Report:
               precision    recall  f1-score   support

 NotMelanoma       0.94      0.98      0.96      1780
    Melanoma       0.98      0.94      0.96      1781

    accuracy                           0.96      3561
   macro avg       0.96      0.96      0.96      3561
weighted avg       0.96      0.96      0.96      3561

